## Scrapping molecules

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time
from tqdm.auto import tqdm

def scrape_molecules_by_id(start_id, end_id):
    base_url = "https://www.vidal.ru/drugs/molecule/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    results = []

    print(f"Starting scrape from ID {start_id} to {end_id}...")

    # Using tqdm for progress tracking
    for i in tqdm(range(start_id, end_id + 1), desc="Scraping molecules"):
        url = f"{base_url}{i}"
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')

                schema_div = soup.find('div', class_='schema')
                if schema_div:
                    h1_tag = schema_div.find('h1')
                    if h1_tag:
                        span_tag = h1_tag.find('span')
                        if span_tag:
                            latin_name = span_tag.get_text(strip=True).strip('()')
                            results.append({'id': i, 'latin_name': latin_name})
            elif response.status_code == 404:
                continue

            time.sleep(0.2) # Slightly reduced delay since it's a large range

        except Exception as e:
            print(f"\nError fetching ID {i}: {e}")

    # Save results to CSV
    with open('molecules_by_id.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'latin_name'])
        writer.writeheader()
        writer.writerows(results)

    print(f"\nFinished! Saved {len(results)} molecules to molecules_by_id.csv")

# Scrape the full range
scrape_molecules_by_id(1, 4000)

Starting scrape from ID 1 to 4000...


Scraping molecules:   0%|          | 0/4000 [00:00<?, ?it/s]


Finished! Saved 2805 molecules to molecules_by_id.csv


## Getting SMILES from molecules names

In [ ]:
!pip install pubchempy

In [ ]:
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem

import pandas as pd
import time
from tqdm.auto import tqdm

In [ ]:
def get_smiles_from_pubchem(molecule_name):
    try:
        # Search for compounds by name. The 'compounds' method returns a list of matching compounds.
        # Take the first result as the most relevant one.
        compounds = pcp.get_compounds(molecule_name, 'name')
        if compounds:
            # If compounds are found, return the canonical SMILES of the first one.
            return compounds[0].connectivity_smiles
        else:
            return None
    except Exception as e:
        print(f"An error occurred while querying PubChem for '{molecule_name}': {e}")
        return None

In [ ]:
# Load the dataset
input_file = 'molecules_by_id.csv'
output_file = 'molecules_processed.csv'

try:
    df_input = pd.read_csv(input_file)
    print(f"Loaded {len(df_input)} molecules from {input_file}")

    results = []

    # Iterate through the rows and process names
    for index, row in tqdm(df_input.iterrows(), total=df_input.shape[0], desc="Processing molecules"):
        name = row['latin_name']
        pubchem_smiles = get_smiles_from_pubchem(name)

        if pubchem_smiles:
            mol = Chem.MolFromSmiles(pubchem_smiles)
            if mol:
                # Generate canonical SMILES
                can_smiles = Chem.MolToSmiles(mol, canonical=True)
                # Generate one random non-canonical SMILES
                non_can_smiles = Chem.MolToSmiles(mol, canonical=False, doRandom=True)

                # Only add to results if we successfully got SMILES
                results.append({
                    'latin_name': name,
                    'canonical_smiles': can_smiles,
                    'non_canonical_smiles': non_can_smiles
                })

        # API rate limit safety
        time.sleep(0.2)

    # Create a new dataframe directly from the successful results
    result_df = pd.DataFrame(results)

    # Save to CSV
    result_df.to_csv(output_file, index=False, encoding='utf-8')

    print(f"\nSuccess! Saved results to {output_file}")
    print(f"Kept {len(result_df)} molecules found in PubChem.")

except Exception as e:
    print(f"An error occurred: {e}")

Loaded 2760 molecules from molecules_by_id.csv


Processing molecules:   0%|          | 0/2760 [00:00<?, ?it/s]

[21:43:40] WARNING: not removing hydrogen atom without neighbors
[21:43:40] WARNING: not removing hydrogen atom without neighbors
[21:44:35] WARNING: not removing hydrogen atom without neighbors
[21:44:35] WARNING: not removing hydrogen atom without neighbors
[21:45:34] WARNING: not removing hydrogen atom without neighbors
[21:45:34] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors



Success! Saved results to molecules_processed.csv
Kept 1997 molecules found in PubChem.


## Getting SMILES with filtering molecules

In [ ]:
def get_heavy_atom_count(mol):
    """Returns the number of heavy (non-hydrogen) atoms in a molecule."""
    if mol:
        return mol.GetNumHeavyAtoms()
    return 0

In [ ]:
import pandas as pd
import time
from tqdm.auto import tqdm
from rdkit.Chem import Descriptors

# Configuration
input_file = 'molecules_by_id.csv'
output_file = 'molecules_filtered.csv'
MAX_HEAVY_ATOMS = 50  # Adjust this threshold to filter polymers/large molecules
MAX_MW = 900          # Molecular Weight threshold

try:
    df_input = pd.read_csv(input_file)
    results = []

    for index, row in tqdm(df_input.iterrows(), total=df_input.shape[0], desc="Filtering molecules"):
        name = row['latin_name']
        pubchem_smiles = get_smiles_from_pubchem(name)

        if pubchem_smiles:
            mol = Chem.MolFromSmiles(pubchem_smiles)
            if mol:
                mw = Descriptors.MolWt(mol)
                heavy_atoms = get_heavy_atom_count(mol)

                # Filter logic: Must be under MW limit AND Heavy Atom limit
                if mw < MAX_MW and heavy_atoms < MAX_HEAVY_ATOMS:
                    results.append({
                        'latin_name': name,
                        'canonical_smiles': Chem.MolToSmiles(mol, canonical=True),
                        'non_canonical_smiles': Chem.MolToSmiles(mol, canonical=False, doRandom=True)
                    })

        time.sleep(0.2)  # Rate limiting

    result_df = pd.DataFrame(results)
    result_df.to_csv(output_file, index=False)
    print(f"\nProcessed {len(df_input)} molecules. Kept {len(result_df)} after filtering.")
    display(result_df.head())

except Exception as e:
    print(f"Error: {e}")

Filtering molecules:   0%|          | 0/2760 [00:00<?, ?it/s]

[22:14:54] WARNING: not removing hydrogen atom without neighbors
[22:14:54] WARNING: not removing hydrogen atom without neighbors
[22:15:49] WARNING: not removing hydrogen atom without neighbors
[22:15:49] WARNING: not removing hydrogen atom without neighbors
[22:16:51] WARNING: not removing hydrogen atom without neighbors
[22:16:51] WARNING: not removing hydrogen atom without neighbors
[22:18:09] WARNING: not removing hydrogen atom without neighbors
[22:18:09] WARNING: not removing hydrogen atom without neighbors



Processed 2760 molecules. Kept 1825 after filtering.


,latin_name,canonical_smiles,mw,heavy_atom_count
0,ACARBOSE,CC1OC(OC2C(CO)OC(OC(C(O)CO)C(O)C(O)C=O)C(O)C2O...,645.61,44
1,ACECLIDINE,CC(=O)OC1CN2CCC1CC2,169.22,12
2,ACENOCOUMAROL,CC(=O)CC(c1ccc([N+](=O)[O-])cc1)c1c(O)c2ccccc2...,353.33,26
3,ACETARSOL,CC(=O)Nc1cc([As](=O)(O)O)ccc1O,275.09,15
4,ACETAZOLAMIDE,CC(=O)Nc1nnc(S(N)(=O)=O)s1,222.25,13


## Picking random molecules

In [ ]:
# Pick 50 random molecules from the processed results
if not result_df.empty:
    random_50 = result_df.sample(n=min(50, len(result_df)))
    random_50.to_csv('50random.csv', index=False, encoding='utf-8')
else:
    print("The results dataframe is empty.")